> **Repository execution note.** Start JupyterLab from the **repository root** and place authorised source data in that same root directory (see [README](../README.md) and [Data requirements](../data/README.md)). These notebooks were originally distributed as standalone packages: any older instructions referring to an included ZIP, a `test/` folder, or bundled data do not apply to this GitHub repository. The notebooks preserve their original input schemas and experiment configurations. Results cached in the original files are intentionally excluded from this public-ready copy.

# KBS — Pandemic, Extreme Values, and Temporal Generalization

### Regression Experiments for Labour Productivity in Independent Portuguese Hotels · 2018–2024

**Objective:** Investigate *why* the models exhibited high prediction errors and how the 2020 disruption and extreme observations affect the ability to predict 2023/2024.

This self-contained notebook is annotated for readers without specialized Machine Learning expertise. It uses the same seven hotel characteristics as the previous notebook, with year as an eighth predictor; **it does not modify the Excel file**.

> **Hypotheses, not conclusions:** (H1) Productivity in 2020 differs from other years; (H2) some extreme observations destabilize the estimators; (H3) past productivity adds information for predicting the following year. The experiments below are intended to test these hypotheses. They do not establish causality.

**Methodological safeguards:** Chronological splitting is mandatory; filters and target-treatment thresholds are computed from training data only; 2023 is used to compare configurations; 2024 is reserved for final evaluation. Errors observed in 2023 are never used to modify the true 2024 values.


## Quick Guide — What Luís Needs to Do

1. Place this `.ipynb` file **and** the Excel file `2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx` in the same folder.
2. In the Jupyter Python environment, install:

   ```bash
   pip install numpy pandas matplotlib scikit-learn openpyxl lxml ipykernel
   pip install xgboost lightgbm catboost
   ```
3. Open the notebook in Jupyter and select **Kernel → Restart Kernel and Run All Cells**.
4. First examine the 2018–2024 plots, then the **2023 Validation** table. The **2024 Test** metrics appear at the end.
5. Open `resultados_kbs_pandemia/`, which contains all CSV files, PNG figures, and a configuration JSON file.

If `xgboost`, `lightgbm`, or `catboost` is not installed, the notebook issues a warning and continues with the available models. To run a faster initial test, change the `MODELOS` list in the next cell.

**Useful terms:** *Training* = years used for learning; *validation* = year used to choose a configuration; *final test* = year reserved for measuring generalization. `MAE` and `RMSE` are error measures (lower is better); `R²` compares performance with the evaluation-set mean (higher is better; it may be negative).


## 1. Configuring Paths, Years, and Models


In [ ]:
from pathlib import Path
import json, sys, platform, importlib.util, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

SEMENTE = 42
FICHEIRO_DADOS = Path('2018- 2024 Luis Silvano - Doctoral Thesis Dataset.xlsx')
PASTA_RESULTADOS = Path('resultados_kbs_pandemia')
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
ANO_TREINO_FINAL = 2022
ANO_VALIDACAO = 2023
ANO_TESTE = 2024

# These four models are optional and may be removed from the list.
# RF and boosting libraries use the CPU; parallelism is limited for Jupyter stability.
MODELOS = ['random_forest', 'lightgbm', 'xgboost', 'catboost']
POLITICAS_2020 = ['todos_os_anos', 'peso_2020_025', 'sem_2020', 'somente_2021_em_diante']
TRATAMENTOS_ALVO = ['original', 'limite_superior_p995', 'log1p']
METRICA_SELECAO = 'val_RMSE'  # Change this only BEFORE running the experiments or inspecting the results.

plt.rcParams.update({'figure.dpi': 105, 'savefig.dpi': 170})
pd.set_option('display.max_columns', 25)

assert ANO_TREINO_FINAL < ANO_VALIDACAO < ANO_TESTE
assert METRICA_SELECAO in {'val_RMSE','val_MAE'}
print('Jupyter working directory:', Path.cwd())
print('Expected Excel file:', FICHEIRO_DADOS.resolve())
print('Requested models:', MODELOS)

## 2. What Are We Predicting, and Which Features Are Allowed?

**`f_labprodct`** is annual labour productivity. Each observation represents one hotel in one year, not a class (0 or 1).

- **Structure:** Capacity, number of accommodation units, and star rating.
- **Spatial context:** Distance to the nearest beach, castle, and church, plus restaurants/cafés within 800 metres.
- **Time:** `f_year` (observation year).

**Important:** Financial variables and same-year employee counts **are not included as predictors**, because they may be components of productivity itself or may only become available after the forecast period. We use `f_number_of_employees` and `f_operating_revenue_turnover2` only to **audit extreme cases**, not as predictors.

Structural and spatial characteristics appear relatively stable. Before submission, the team must document **when they were actually measured**; if they were collected after the forecast year, historical versions must be reconstructed. The code cannot automatically resolve this issue.


In [ ]:
ID_HOTEL='ht_id'
COL_ANO='f_year'
COL_ALVO='f_labprodct'
COL_CADEIA='ht_hotel_chain_d'
ATRIBUTOS_ESTRUTURAIS = [
    'ht_capacity', 'ht_log_number_of_accommodation_units',
    'ht_stars_classification',
]
ATRIBUTOS_ESPACIAIS = [
    'ht_location_beach1', 'ht_location_castle1',
    'ht_location_church1', 'ht_Restaurants&CoffesL800m',
]
ATRIBUTOS_HOTEL = ATRIBUTOS_ESTRUTURAIS + ATRIBUTOS_ESPACIAIS
ATRIBUTOS = ATRIBUTOS_HOTEL + [COL_ANO]
COLUNAS_AUDITORIA=['f_number_of_employees', 'f_operating_revenue_turnover2']
# Reuse the validated loader, including two additional columns for outlier auditing.
ATRIBUTOS_HOTEL = ATRIBUTOS_HOTEL + COLUNAS_AUDITORIA

print('Target:', COL_ALVO)
print('Actual predictors (8):', ATRIBUTOS)
print('For auditing only (NOT included in the model):', COLUNAS_AUDITORIA)

## 3. Loading the Actual Excel File and Auditing the Panel

This cell reuses the selective-loading method already tested in the original notebook. It displays the full path, checks the `New Dataset` worksheet, and requires the correct columns. It does not silently substitute column names.

Reading the Excel file takes longer than later steps because the original file contains millions of cells. If `lxml` is installed, only the required columns are extracted.


In [ ]:
# This cell explicitly checks WHICH file was read and WHICH worksheet was selected.
# The fast loader reads only the required columns and does not modify the original Excel file.
import re
import zipfile
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

FOLHA_DADOS = 'New Dataset'
COLUNAS_NECESSARIAS = list(dict.fromkeys(
    [ID_HOTEL, COL_ANO, COL_ALVO, COL_CADEIA] + ATRIBUTOS_HOTEL
))

if not FICHEIRO_DADOS.exists():
    raise FileNotFoundError(
        f'Excel file not found: {FICHEIRO_DADOS.resolve()}\n'
        'Place the Excel file next to this notebook or edit FICHEIRO_DADOS in Section 1.\n'
        f'Current Jupyter directory: {Path.cwd()}'
    )

print('Current Jupyter directory:', Path.cwd())
print('File loaded (absolute path):', FICHEIRO_DADOS.resolve())
print('File size (MB):', round(FICHEIRO_DADOS.stat().st_size / 1024**2, 2))

if FICHEIRO_DADOS.suffix.lower() in ('.xlsx', '.xlsm'):
    pasta_excel = load_workbook(FICHEIRO_DADOS, read_only=True, data_only=True)
    folhas = pasta_excel.sheetnames
    print('Worksheets found:', folhas)
    if FOLHA_DADOS not in folhas:
        pasta_excel.close()
        raise ValueError(
            f'Worksheet {FOLHA_DADOS!r} does not exist in this workbook. '
            f'Available worksheets: {folhas}. Verify that you selected the correct file.'
        )
    folha = pasta_excel[FOLHA_DADOS]
    cabecalho_excel = [str(v).strip() if v is not None else ''
                       for v in next(folha.iter_rows(min_row=1, max_row=1, values_only=True))]
    pasta_excel.close()
    print('Columns in the original Excel file:', len(cabecalho_excel))

    colunas_em_falta = [c for c in COLUNAS_NECESSARIAS if c not in cabecalho_excel]
    if colunas_em_falta:
        print('EXAMPLES OF EXISTING COLUMNS:', cabecalho_excel[:25])
        raise ValueError(
            'The following columns were not found in the OPENED file:\n  - '
            + '\n  - '.join(colunas_em_falta)
            + '\nCheck the absolute path above and the Excel file version. '
              'These columns are present in the original Excel file supplied by the team. '
              'We will not silently replace these variables with others.'
        )

    def read_selected_excel_columns(caminho, nomes, cabecalho):
        """Read only the requested columns without processing millions of cells in pandas.

        If lxml is unavailable, fall back to pandas.read_excel.
        Do not modify formulas or values in the original Excel file.
        """
        try:
            from lxml import etree as ET
        except ImportError:
            print('lxml is not installed; standard loading may take several minutes.')
            return pd.read_excel(
                caminho, sheet_name=FOLHA_DADOS, usecols=lambda c: c.strip() in nomes,
                engine='openpyxl'
            )

        # This project version contains a single worksheet. For workbooks with multiple
        # worksheets, the standard loader selects the specified sheet unambiguously.
        if len(folhas) != 1:
            print('Workbook with multiple worksheets: reading through pandas.')
            return pd.read_excel(
                caminho, sheet_name=FOLHA_DADOS, usecols=lambda c: c.strip() in nomes,
                engine='openpyxl'
            )

        ns = '{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
        # Map Excel column letters (e.g., 'A', 'HQ') to variable names.
        letra_para_nome = {
            get_column_letter(i+1): nome
            for i, nome in enumerate(cabecalho)
            if nome in nomes
        }
        linhas = []
        with zipfile.ZipFile(caminho) as arq:
            texto_compartilhado = []
            if 'xl/sharedStrings.xml' in arq.namelist():
                raiz = ET.parse(arq.open('xl/sharedStrings.xml')).getroot()
                texto_compartilhado = [
                    ''.join(si.itertext()) for si in raiz.findall(ns + 'si')
                ]
            with arq.open('xl/worksheets/sheet1.xml') as fluxo:
                parser = ET.iterparse(fluxo, events=('end',), tag=ns+'row', huge_tree=True)
                for _, linha in parser:
                    if linha.get('r') != '1':
                        registo = {}
                        for celula in linha.iterchildren(tag=ns+'c'):
                            # Cell A12 -> column A; the row number is not required.
                            letra = celula.get('r', '').rstrip('0123456789')
                            nome = letra_para_nome.get(letra)
                            if nome is None:
                                continue
                            valor = celula.findtext(ns+'v')
                            if valor is None:
                                continue
                            tipo = celula.get('t')
                            if tipo == 's':
                                valor = texto_compartilhado[int(valor)]
                            elif tipo == 'b':
                                valor = (valor == '1')
                            elif tipo not in ('str', 'inlineStr'):
                                try:
                                    valor = float(valor)
                                except (ValueError, TypeError):
                                    pass
                            registo[nome] = valor
                        linhas.append(registo)
                    # Avoid retaining all three million Excel cells in memory.
                    linha.clear()
                    while linha.getprevious() is not None:
                        del linha.getparent()[0]
        return pd.DataFrame(linhas, columns=nomes)

    dados_brutos = read_selected_excel_columns(
        FICHEIRO_DADOS, COLUNAS_NECESSARIAS, cabecalho_excel
    )
elif FICHEIRO_DADOS.suffix.lower() == '.csv':
    cabecalho_excel = pd.read_csv(FICHEIRO_DADOS, nrows=0).columns.tolist()
    dados_brutos = pd.read_csv(FICHEIRO_DADOS, usecols=COLUNAS_NECESSARIAS)
else:
    raise ValueError('Unrecognized format. Use .xlsx, .xlsm, or .csv.')

dados_brutos.columns = [str(c).strip() for c in dados_brutos.columns]
print('Worksheet used:', FOLHA_DADOS if FICHEIRO_DADOS.suffix.lower() != '.csv' else 'CSV')
print('Rows read:', len(dados_brutos))
print('Columns loaded for the study:', list(dados_brutos.columns))
print('NOTE: only the relevant variables are loaded; the original Excel file has 277 columns.')
display(dados_brutos.head(3))

In [ ]:
# Data conversion is explicit: future values are not used to compute thresholds.
necessarias = list(dict.fromkeys([ID_HOTEL, COL_ANO, COL_ALVO, COL_CADEIA] +
                                 ATRIBUTOS + COLUNAS_AUDITORIA))
faltam = [c for c in necessarias if c not in dados_brutos.columns]
if faltam:
    raise ValueError('The following variables were not found in the opened Excel file: ' + repr(faltam))
painel = dados_brutos[necessarias].copy()
for c in [COL_ANO, COL_CADEIA, COL_ALVO] + ATRIBUTOS_HOTEL:
    painel[c] = pd.to_numeric(painel[c], errors='coerce')
painel = painel.replace([np.inf, -np.inf], np.nan)
qtd_original=len(painel)
painel = painel.loc[(painel[COL_CADEIA] == 0) & (painel[COL_ANO].between(2018, 2024))].copy()
qtd_indep=len(painel)
sem_alvo=int(painel[COL_ALVO].isna().sum())
painel = painel.dropna(subset=[ID_HOTEL, COL_ANO, COL_ALVO]).copy()
if (painel[COL_ALVO] < 0).any():
    raise ValueError('Negative f_labprodct values were found; review before using log1p.')
painel[COL_ANO] = painel[COL_ANO].astype(int)
painel[ID_HOTEL] = painel[ID_HOTEL].astype('Int64').astype(str)
if painel.duplicated([ID_HOTEL, COL_ANO]).any():
    raise ValueError('Duplicate hotel/year pairs were found. Review the panel construction before training.')
painel = painel.sort_values([ID_HOTEL,COL_ANO]).reset_index(drop=True)
resumo_anos = painel.groupby(COL_ANO).agg(
    observacoes=(COL_ALVO,'size'),
    hoteis=(ID_HOTEL,'nunique'),
    media=(COL_ALVO,'mean'), mediana=(COL_ALVO,'median'),
    p95=(COL_ALVO,lambda s:s.quantile(.95)),
    p99=(COL_ALVO,lambda s:s.quantile(.99)),
    maximo=(COL_ALVO,'max'),
).reset_index()
print(f'Raw rows: {qtd_original:,} | independent (2018–2024): {qtd_indep:,} | '
      f'missing productivity: {sem_alvo:,}')
print(f'Usable rows: {len(painel):,} | distinct hotels: {painel[ID_HOTEL].nunique():,}')
display(resumo_anos.round(2))
assert set(painel[COL_ANO].unique()) == set(range(2018,2025))

## 4. Exploratory Diagnostics — What Changed During the Pandemic?

**The mean/median plot is descriptive.** The composition of observed hotels may change across years; we therefore also compare 2019 with 2020 **for the same hotels**. This reduces the risk of mistaking changes in sample composition for actual changes.

The distribution plot focuses on the centre of productivity; extreme values are not removed from the data or evaluation. The axis is only limited to make the figure readable.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(resumo_anos[COL_ANO], resumo_anos['media'], marker='o', label='Mean')
axes[0].plot(resumo_anos[COL_ANO], resumo_anos['mediana'], marker='s', label='Median')
axes[0].axvspan(2019.5, 2020.5, alpha=.13, color='gray', label='Year 2020')
axes[0].set(xlabel='Year', ylabel='Labour productivity', title='Annual productivity trends')
axes[0].legend()
axes[0].grid(alpha=.2)

# Compare the same hotels in 2019 and 2020, without including different entrants.
pares = painel[painel[COL_ANO].isin([2019,2020])].pivot(index=ID_HOTEL, columns=COL_ANO, values=COL_ALVO).dropna()
if len(pares):
    dif=pares[2020]-pares[2019]
    axes[1].hist(dif.clip(dif.quantile(.01),dif.quantile(.99)),bins=32)
    axes[1].axvline(0,color='black',linestyle='--',linewidth=1)
    axes[1].set(xlabel='2020 minus 2019 (extremes clipped in the plot only)',
                ylabel='Number of hotels', title=f'Within-hotel change (n={len(pares)})')
    print(f'Hotels observed in both years: {len(pares)} | median change: {dif.median():.2f} | '
          f'fraction with a decline: {(dif<0).mean():.1%}')
plt.tight_layout()
fig.savefig(PASTA_RESULTADOS/'figura_01_pandemia_mudanca_anual.png',bbox_inches='tight')
plt.show()

fig,ax=plt.subplots(figsize=(10,4.5))
limite_visual=np.nanquantile(painel[COL_ALVO],.98)
series=[painel.loc[painel[COL_ANO]==ano,COL_ALVO].clip(upper=limite_visual).to_numpy() for ano in range(2018,2025)]
ax.boxplot(series, tick_labels=list(range(2018,2025)), showfliers=False)
ax.set(xlabel='Year',ylabel='Productivity (axis limited to p98 for visualization)',
       title='Annual dispersion: same axis, no records removed from analysis')
ax.grid(axis='y',alpha=.2)
fig.tight_layout()
fig.savefig(PASTA_RESULTADOS/'figura_02_distribuicao_anual.png',bbox_inches='tight')
plt.show()

## 5. Before Training: Split the Years and Inspect Extreme Values

- Training for **configuration selection:** 2018–2022.
- Validation: **2023**, used to choose the model, the 2020 policy, and the target treatment.
- Final test: **2024**. We do not use it to select configurations.

Observations for a given hotel may be repeated in different years. This is permitted here because the goal is to predict **future years for known hotels**. Evaluating entirely new hotels requires splitting by `ht_id` and constitutes a different protocol.

A common mistake would be to remove high-productivity hotels from 2023/2024, artificially reducing RMSE. **We do not do this.** All robust processing is applied to the **training target**, and errors are always measured against the true 2023/2024 values.


In [ ]:
treino_base = painel[painel[COL_ANO] <= ANO_TREINO_FINAL].copy()
validacao = painel[painel[COL_ANO] == ANO_VALIDACAO].copy()
teste_final = painel[painel[COL_ANO] == ANO_TESTE].copy()
assert treino_base[COL_ANO].max() < validacao[COL_ANO].min() < teste_final[COL_ANO].min()
assert all(len(x)>0 for x in [treino_base, validacao, teste_final])
contagens = pd.DataFrame([
 {'conjunto':'treino','anos':'2018–2022','observacoes':len(treino_base),'hoteis':treino_base[ID_HOTEL].nunique()},
 {'conjunto':'validacao','anos':'2023','observacoes':len(validacao),'hoteis':validacao[ID_HOTEL].nunique()},
 {'conjunto':'teste','anos':'2024','observacoes':len(teste_final),'hoteis':teste_final[ID_HOTEL].nunique()},
])
display(contagens)

print('Auditing example (NOT used as a predictor):')
cols=[ID_HOTEL,COL_ANO,COL_ALVO,'f_number_of_employees','f_operating_revenue_turnover2']
display(painel.loc[painel[ID_HOTEL].isin(['1824','721']),cols].sort_values([ID_HOTEL,COL_ANO]))
print('p99.5 of productivity in the initial training set:',round(treino_base[COL_ALVO].quantile(.995),3))

## 6. Four Policies for the Year 2020

Each policy represents a **different scientific hypothesis**. The composition of the **2023 validation set remains exactly the same** across comparisons.

| Policy | What changes in training | Which hypothesis does it examine? |
|---|---|---|
| `todos_os_anos` | Train on 2018–2022, including 2020 normally | Reference comparison |
| `peso_2020_025` | Keep 2020, but weight each observation from that year at 25% | Reduce the impact of the disruption without dropping observations |
| `sem_2020` | Train on 2018, 2019, 2021, and 2022 | Does omitting the atypical period help? |
| `somente_2021_em_diante` | Use 2021–2022 when fitting for validation | More recent data, but fewer observations and no pre-pandemic years |

**Caution:** Excluding 2020 is not an automatic correction. The smaller sample may worsen generalization; the data must determine whether it helps. The `somente_2021_em_diante` policy also changes the sample size, so it does not isolate the pandemic effect.


## 7. Three Ways to Handle Extreme Values — During Training Only

- **Original:** Leave productivity unchanged.
- **Upper cap at p99.5:** Values above the **training-set** 99.5th percentile are replaced by that threshold **during fitting**. Validation and test targets and errors remain on the original scale. No hotels are removed from the study.
- **log1p:** Fit the model to `log(1 + productivity)`, then convert predictions back to the original scale with `expm1`. This is valid for the nonnegative values verified in the dataset.

The p99.5 threshold is an initial experimental choice, **not a statistical truth**. Before publication, sensitivity to other thresholds—such as p99 and p99.75—and robust regression objectives should be examined.


In [ ]:
def select_training_data_by_policy(df, politica):
    if politica == 'todos_os_anos':
        out=df.copy()
    elif politica == 'peso_2020_025':
        out=df.copy()
    elif politica == 'sem_2020':
        out=df.loc[df[COL_ANO] != 2020].copy()
    elif politica == 'somente_2021_em_diante':
        out=df.loc[df[COL_ANO] >= 2021].copy()
    else:
        raise ValueError('Unknown policy: '+str(politica))
    if out.empty:
        raise ValueError('Empty training set for policy '+politica)
    pesos = np.where((out[COL_ANO].to_numpy() == 2020) &
                     (politica=='peso_2020_025'), .25, 1.0).astype(float)
    return out, pesos


def build_regressor(nome):
    if nome=='random_forest':
        return RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                     n_jobs=2, random_state=SEMENTE)
    if nome=='lightgbm':
        from lightgbm import LGBMRegressor
        return LGBMRegressor(n_estimators=500, learning_rate=.05, num_leaves=15,
                             min_child_samples=25, random_state=SEMENTE,
                             verbosity=-1, n_jobs=2)
    if nome=='xgboost':
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=500, learning_rate=.05, max_depth=4,
                            subsample=.9, colsample_bytree=.9,
                            objective='reg:squarederror',random_state=SEMENTE,n_jobs=1)
    if nome=='catboost':
        from catboost import CatBoostRegressor
        return CatBoostRegressor(iterations=500, learning_rate=.05, depth=5,
                                 loss_function='RMSE', random_seed=SEMENTE,
                                 verbose=False, thread_count=2,
                                 allow_writing_files=False)
    raise ValueError('Unknown model: '+str(nome))


def calculate_metrics(y, pred, prefix=''):
    y=np.asarray(y,dtype=float); pred=np.asarray(pred,dtype=float)
    if not np.isfinite(pred).all():
        raise ValueError('The model returned non-finite predictions')
    return {prefix+'MAE':float(mean_absolute_error(y,pred)),
            prefix+'RMSE':float(np.sqrt(mean_squared_error(y,pred))),
            prefix+'R2':float(r2_score(y,pred))}


def train_model(nome, df_treino, df_prever, politica, tratamento, colunas):
    tr, pesos=select_training_data_by_policy(df_treino,politica)
    imp=SimpleImputer(strategy='median')
    # Imputation statistics are computed exclusively from training observations.
    if any(tr[c].notna().sum()==0 for c in colunas):
        raise ValueError('A feature has no non-missing values in the selected training data.')
    Xtr=pd.DataFrame(imp.fit_transform(tr[colunas].astype(float)), columns=colunas, index=tr.index)
    Xp=pd.DataFrame(imp.transform(df_prever[colunas].astype(float)), columns=colunas, index=df_prever.index)
    ytr=tr[COL_ALVO].to_numpy(dtype=float)
    limiar=None
    if tratamento=='original':
        y_ajuste=ytr
    elif tratamento=='limite_superior_p995':
        limiar=float(np.quantile(ytr,.995))
        y_ajuste=np.minimum(ytr,limiar)
    elif tratamento=='log1p':
        if (ytr<0).any(): raise ValueError('log1p requires nonnegative productivity')
        y_ajuste=np.log1p(ytr)
    else:
        raise ValueError('Unknown target treatment: '+str(tratamento))
    modelo=build_regressor(nome)
    modelo.fit(Xtr,y_ajuste,sample_weight=pesos)
    pred=np.asarray(modelo.predict(Xp)).reshape(-1)
    if tratamento=='log1p':
        pred=np.expm1(pred)
    return pred, {'modelo':modelo, 'imputador':imp, 'limite_y_treino':limiar,
                  'n_treino':len(tr), 'n_ano_2020':int((tr[COL_ANO]==2020).sum())}

modelos_disponiveis=[]
for nome in MODELOS:
    modulo = {'random_forest':'sklearn','lightgbm':'lightgbm','xgboost':'xgboost','catboost':'catboost'}[nome]
    if importlib.util.find_spec(modulo): modelos_disponiveis.append(nome)
    else: print('Warning:',nome,'not run: library',modulo,'not installed')
if not modelos_disponiveis:
    raise RuntimeError('No models available! Install scikit-learn and rerun.')
print('Models actually available:', modelos_disponiveis)

## 8. Factorial Experiment on 2023 Validation Data

We run all **model × 2020 policy × extreme-value treatment** combinations, always predicting **the same 2023 observations**. The configuration with the lowest 2023 RMSE is selected (the predefined primary metric).

**2024 is not accessed as ground truth at this stage.** This is essential: selecting configurations based on 2024 errors would yield an overly optimistic estimate. The 2024 data remain in a separate variable, used only at the end.

> Total runtime depends on CPU performance. To reduce the initial runtime, first run with `MODELOS=['random_forest','lightgbm']`, then repeat using the full list; both require restarting the kernel.


In [ ]:
resultados_validacao=[]
predicoes_validacao={}
falhas=[]
inicio_global=time.perf_counter()
y_val=validacao[COL_ALVO].to_numpy(dtype=float)
quantis_y=[.5,.9,.95]
for nome in modelos_disponiveis:
    for politica in POLITICAS_2020:
        for tratamento in TRATAMENTOS_ALVO:
            chave=f'{nome}|{politica}|{tratamento}'
            inicio=time.perf_counter()
            try:
                pred, info=train_model(nome,treino_base,validacao,politica,tratamento,ATRIBUTOS)
                linha={'modelo':nome,'politica_2020':politica,'tratamento_alvo':tratamento,
                       'n_treino':info['n_treino'],'n_2020':info['n_ano_2020'],
                       'limite_superior_treino':info['limite_y_treino'],
                       'segundos':round(time.perf_counter()-inicio,2)}
                linha.update(calculate_metrics(y_val,pred,'val_'))
                resultados_validacao.append(linha)
                predicoes_validacao[chave]=pred
            except Exception as ex:
                falhas.append({'modelo':nome,'politica':politica,'tratamento':tratamento,
                               'erro':f'{type(ex).__name__}: {ex}'})
                print('[FAILED]',chave,':',ex)
    print('Completed:',nome, 'total time:',round(time.perf_counter()-inicio_global,1),'s')

if not resultados_validacao:
    raise RuntimeError('No experiment completed successfully. See the errors.')
resultados_validacao=pd.DataFrame(resultados_validacao).sort_values(
    [METRICA_SELECAO,'val_MAE']).reset_index(drop=True)
print(f'Configurations tested: {len(resultados_validacao)} out of '
      f'{len(modelos_disponiveis)*len(POLITICAS_2020)*len(TRATAMENTOS_ALVO)}')
print('Best configurations on 2023 VALIDATION:')
display(resultados_validacao.head(15).round(3))
if falhas: display(pd.DataFrame(falhas))

## 9. What Actually Improved on Validation?

A ranking alone is insufficient. The tables below show whether a given 2020 policy helps **within the same model and target treatment**, or whether improvements mostly arise from capping extreme values.

The plot shows 2023 RMSE. **Lower is better.** No 2024 test values are used in this comparison.


In [ ]:
resumo_politicas = resultados_validacao.pivot_table(
    index=['modelo','tratamento_alvo'], columns='politica_2020',
    values='val_RMSE',aggfunc='first').round(2)
display(resumo_politicas)

# One heatmap per model is clearer than dozens of bars.
# Shared colour scale: the same colour indicates approximately the same RMSE.
nomes_politicas={
    'todos_os_anos':'All years',
    'peso_2020_025':'2020 weight 0.25',
    'sem_2020':'Exclude 2020',
    'somente_2021_em_diante':'From 2021 onward',
}
nomes_trat={'original':'Original', 'limite_superior_p995':'p99.5 (training)', 'log1p':'log1p'}
fig, axes=plt.subplots(2,2,figsize=(12,8.2),constrained_layout=True)
ax_lista=axes.flat
im=None
for j,modelo in enumerate(modelos_disponiveis[:4]):
    ax=ax_lista[j]
    base=resultados_validacao.loc[resultados_validacao['modelo']==modelo]
    matrix=base.pivot(index='politica_2020',columns='tratamento_alvo',values='val_RMSE')
    matrix=matrix.reindex(index=POLITICAS_2020,columns=TRATAMENTOS_ALVO)
    im=ax.imshow(matrix.to_numpy(),aspect='auto',cmap='YlGnBu',vmin=35,vmax=110)
    ax.set_title(modelo.replace('_',' ').title(),fontweight='semibold')
    ax.set_xticks(range(len(TRATAMENTOS_ALVO)),[nomes_trat[t] for t in TRATAMENTOS_ALVO],rotation=18,ha='right')
    ax.set_yticks(range(len(POLITICAS_2020)),[nomes_politicas[t] for t in POLITICAS_2020])
    for rr in range(len(POLITICAS_2020)):
        for cc in range(len(TRATAMENTOS_ALVO)):
            valor=matrix.iloc[rr,cc]
            if pd.notna(valor):
                ax.text(cc,rr,f'{valor:.1f}',ha='center',va='center',
                        color='white' if valor>75 else 'black',fontsize=10)
for k in range(len(modelos_disponiveis),4): ax_lista[k].axis('off')
fig.colorbar(im,ax=axes.ravel().tolist(),shrink=.73,label='2023 RMSE (lower is better)')
fig.suptitle('Impact of 2020 policies and target robustness on 2023 validation',fontsize=13)
fig.savefig(PASTA_RESULTADOS/'figura_03_politicas_e_robustez_validacao.png',bbox_inches='tight')
plt.show()

vencedor=resultados_validacao.iloc[0].to_dict()
print('CONFIGURATION SELECTED USING 2023 ONLY')
print('  Model:',vencedor['modelo'])
print('  2020 policy:',vencedor['politica_2020'])
print('  Productivity treatment:',vencedor['tratamento_alvo'])
print('  Validation RMSE:',round(vencedor['val_RMSE'],3))

## 10. Diagnosing the Cases That Contributed Most to 2023 Error

To understand RMSE, we identify the hotels with the largest differences between observed and predicted productivity. This **does not justify excluding these hotels**. The objective is to guide the review of employee counts, turnover, zero handling, and the accounting definition of productivity.

The table includes two 2023 scenarios: the selected configuration and the best conventional training result (all years + original target). Their comparison helps determine whether the robust approach merely reduced the influence of an excessively large prediction.


In [ ]:
chave_v=f"{vencedor['modelo']}|{vencedor['politica_2020']}|{vencedor['tratamento_alvo']}"
base_conv=resultados_validacao[(resultados_validacao['politica_2020']=='todos_os_anos') &
                              (resultados_validacao['tratamento_alvo']=='original')].iloc[0]
chave_b=f"{base_conv['modelo']}|{base_conv['politica_2020']}|{base_conv['tratamento_alvo']}"
diag=validacao[[ID_HOTEL,COL_ANO,COL_ALVO,'f_number_of_employees',
               'f_operating_revenue_turnover2']].copy()
diag['previsao_vencedor']=predicoes_validacao[chave_v]
diag['previsao_referencia_original']=predicoes_validacao[chave_b]
diag['erro_abs_vencedor']=abs(diag[COL_ALVO]-diag['previsao_vencedor'])
diag['erro2_referencia_original']=(diag[COL_ALVO]-diag['previsao_referencia_original'])**2
print('Ten largest absolute errors for the selected configuration:')
display(diag.sort_values('erro_abs_vencedor',ascending=False).head(10).round(3))
print('Ten largest contributions to RMSE for the original baseline:')
display(diag.sort_values('erro2_referencia_original',ascending=False).head(10).round(3))

## 11. Final 2024 Test — Only After Selecting the Configuration Using 2023

We can now evaluate the held-out year 2024. To emulate a real forecast, **we refit the model using data available through the end of 2023** (2018–2023), while retaining the same weighting/exclusion policy and extreme-value treatment chosen during validation.

Predefined comparators:

1. **Mean Dummy:** Predicts mean productivity in the training data. Provides a minimum-performance reference.
2. **Conventional Random Forest:** Uses all years and the original target; serves as the reference from the initial experiments.
3. **Selected configuration:** Combination of model, 2020 policy, and target treatment chosen exclusively using 2023.

We also measure MAE, RMSE, and R² on the complete 2024 data — **without removing outliers**. We do not select a new winner based on the 2024 results.


In [ ]:
refit_ate_2023 = painel[painel[COL_ANO] <= ANO_VALIDACAO].copy()
y_teste=teste_final[COL_ALVO].to_numpy(dtype=float)
comparadores=[]
predicoes_teste={}

# 1) Dummy model using only data available through 2023.
imp_dummy=SimpleImputer(strategy='median')
X_dummy=imp_dummy.fit_transform(refit_ate_2023[ATRIBUTOS])
dummy=DummyRegressor(strategy='mean')
dummy.fit(X_dummy,refit_ate_2023[COL_ALVO].to_numpy())
pred_dummy=dummy.predict(imp_dummy.transform(teste_final[ATRIBUTOS]))
comparadores.append({'avaliacao':'dummy_media','modelo':'dummy',
                     'politica_2020':'todos_os_anos','tratamento_alvo':'original',
                     **calculate_metrics(y_teste,pred_dummy,'teste_')})
predicoes_teste['dummy_media']=pred_dummy

# 2) Previous baseline with all observations and untransformed productivity.
pred_rf,info_rf=train_model('random_forest',refit_ate_2023,teste_final,
                             'todos_os_anos','original',ATRIBUTOS)
comparadores.append({'avaliacao':'random_forest_original','modelo':'random_forest',
                     'politica_2020':'todos_os_anos','tratamento_alvo':'original',
                     **calculate_metrics(y_teste,pred_rf,'teste_')})
predicoes_teste['random_forest_original']=pred_rf

# 3) Model selected before accessing the test results.
pred_ganhador,info_final=train_model(
    vencedor['modelo'],refit_ate_2023,teste_final,
    vencedor['politica_2020'],vencedor['tratamento_alvo'],ATRIBUTOS)
comparadores.append({'avaliacao':'vencedor_validacao_2023','modelo':vencedor['modelo'],
                     'politica_2020':vencedor['politica_2020'],
                     'tratamento_alvo':vencedor['tratamento_alvo'],
                     **calculate_metrics(y_teste,pred_ganhador,'teste_')})
predicoes_teste['vencedor_validacao_2023']=pred_ganhador

resultados_teste=pd.DataFrame(comparadores)
print('FINAL 2024 TEST (refit through 2023; no test-based selection)')
display(resultados_teste.round(3))
print('The winning model was fixed using 2023; do not change it after seeing the 2024 column.')
# Automatic interpretation, without reselecting the winner after seeing 2024.
ref_2024=float(resultados_teste.loc[resultados_teste['avaliacao']=='random_forest_original','teste_RMSE'].iloc[0])
vencedor_2024=float(resultados_teste.loc[resultados_teste['avaliacao']=='vencedor_validacao_2023','teste_RMSE'].iloc[0])
if vencedor_2024 < ref_2024:
    print(f'In 2024, the configuration selected using 2023 improves RMSE by {ref_2024-vencedor_2024:.2f} compared with conventional Random Forest.')
else:
    print(f'In 2024, the configuration selected using 2023 does NOT outperform conventional Random Forest: RMSE {vencedor_2024:.2f} vs {ref_2024:.2f}.')
    print('This is a valid result: gains in 2023 do not guarantee gains the following year. We must not replace the winner based on the test results.')


## 12. Supplementary Evaluation: Do Spatial Features Really Help?

We retain the selected configuration but train **two models over the same training period**: (A) structural characteristics and year only; (B) structural and spatial characteristics plus year. This allows us to estimate the **incremental predictive value** of collecting distance and local-count features. This is an *ablation*, not a causal test.

> This analysis was prespecified here, but the 2024 data remain a test only, not a criterion for further feature selection. Before finalizing the paper, feature families and metrics should be frozen in a protocol.


In [ ]:
colunas_ab1=ATRIBUTOS_ESTRUTURAIS + [COL_ANO]
colunas_ab2=ATRIBUTOS_ESTRUTURAIS + ATRIBUTOS_ESPACIAIS + [COL_ANO]
res_ablacao=[]
for etiqueta,cols in [('estrutura_e_ano',colunas_ab1),('estrutura_espaco_e_ano',colunas_ab2)]:
    pred,_=train_model(vencedor['modelo'],refit_ate_2023,teste_final,
                          vencedor['politica_2020'],vencedor['tratamento_alvo'],cols)
    res_ablacao.append({'caracteristicas':etiqueta, 'n_colunas':len(cols),
                         **calculate_metrics(y_teste,pred,'teste_')})
resultado_ablacao=pd.DataFrame(res_ablacao)
display(resultado_ablacao.round(3))

## 13. Additional Experiment: **Previous-Year** Productivity (Lag-1)

Geographical attributes change little across years. The following experiment therefore asks whether **productivity from the immediately preceding year** provides additional information. This is a forecasting comparison for **known hotels**, not new ones.

**Avoiding temporal leakage:** To predict 2023, `produtividade_ano_anterior` comes from **2022**; to predict 2024, it comes from **2023**. We never use productivity from the year being predicted or contemporaneous financial variables.

We compare only hotels with a valid observation in the previous year; the same subset is used for both models. We distinguish these results from the main results because the evaluation sample is smaller. We also include a **persistence** baseline, which predicts next year's productivity to be exactly the same as the previous year's.


In [ ]:
# The merge shifts year by +1 and, unlike shift after removing NaN values,
# ensures lag=2022 when predicting 2023 (never incorrectly uses 2021).
lag_anterior=painel[[ID_HOTEL,COL_ANO,COL_ALVO]].copy()
lag_anterior[COL_ANO]=lag_anterior[COL_ANO]+1
lag_anterior=lag_anterior.rename(columns={COL_ALVO:'produtividade_ano_anterior'})
painel_lag=painel.merge(lag_anterior,on=[ID_HOTEL,COL_ANO],how='left',validate='one_to_one')
assert painel_lag[COL_ANO].equals(painel[COL_ANO])

# A hotel-year pair is eligible only if previous-year productivity is observed.
painel_lag=painel_lag[painel_lag['produtividade_ano_anterior'].notna()].copy()
tr_lag_val=painel_lag[painel_lag[COL_ANO] <= 2022].copy()
val_lag=painel_lag[painel_lag[COL_ANO]==2023].copy()
tr_lag_teste=painel_lag[painel_lag[COL_ANO] <= 2023].copy()
te_lag=painel_lag[painel_lag[COL_ANO]==2024].copy()
assert all(not d.empty for d in [tr_lag_val,val_lag,tr_lag_teste,te_lag])
print('Subset with a strict t-1 lag:')
print(' Training through 2022:',len(tr_lag_val),'| validation 2023:',len(val_lag))
print(' Refit through 2023:',len(tr_lag_teste),'| test 2024:',len(te_lag))

cols_sem_lag=ATRIBUTOS
cols_com_lag=ATRIBUTOS+['produtividade_ano_anterior']
res_lag=[]
pred_lag_teste={}
for etapa,treino_l,avaliacao in [
    ('validacao_2023',tr_lag_val,val_lag),('teste_2024',tr_lag_teste,te_lag)
]:
    yobs=avaliacao[COL_ALVO].to_numpy(dtype=float)
    # Essential baseline for checking whether the model beats the previous-year value.
    pred_ingenuo=avaliacao['produtividade_ano_anterior'].to_numpy(dtype=float)
    res_lag.append({'etapa':etapa,'metodo':'persistencia_t_menos_1','observacoes':len(avaliacao),
                    **calculate_metrics(yobs,pred_ingenuo)})
    for label,cols in [('sem_lag',cols_sem_lag),('com_lag_t_menos_1',cols_com_lag)]:
        pred,_=train_model(vencedor['modelo'],treino_l,avaliacao,
                             vencedor['politica_2020'],vencedor['tratamento_alvo'],cols)
        res_lag.append({'etapa':etapa,'metodo':label,'observacoes':len(avaliacao),
                        **calculate_metrics(yobs,pred)})
        if etapa=='teste_2024': pred_lag_teste[label]=pred
resultados_lag=pd.DataFrame(res_lag)
display(resultados_lag.round(3))

# Explanation: a strict t-1 lag is available for only some hotels; therefore,
# these results must NOT be directly compared with the full 2024 test results.
print('CAUTION: interpret the lag-1 experiment as a supplementary analysis. The value of lagged productivity may change substantially between years; do not automatically promote it to the final model.')

## 14. Diagnosing 2024 Residuals

For the final test, we inspect the differences between observed and predicted values. **This diagnostic is not used to change the selected model.** Its purpose is to document limitations and inform subsequent research.


In [ ]:
residuos=teste_final[[ID_HOTEL,COL_ANO,COL_ALVO]].copy()
for nome,pred in predicoes_teste.items():
    residuos['prev_'+nome]=pred
    residuos['erro_abs_'+nome]=abs(residuos[COL_ALVO]-pred)
fig,ax=plt.subplots(figsize=(7,5))
ax.scatter(residuos[COL_ALVO],residuos['prev_vencedor_validacao_2023'],alpha=.4,s=18)
limite=float(max(residuos[COL_ALVO].quantile(.99),residuos['prev_vencedor_validacao_2023'].quantile(.99)))
ax.plot([0,limite],[0,limite],linestyle='--',color='black',linewidth=1)
ax.set(xlim=(0,limite),ylim=(0,limite), xlabel='Observed productivity in 2024',
       ylabel='Predicted productivity', title='Observed vs predicted (visualization limited to p99)')
ax.grid(alpha=.2)
plt.tight_layout()
fig.savefig(PASTA_RESULTADOS/'figura_04_real_vs_previsto_2024.png',bbox_inches='tight')
plt.show()

# The figure is only zoomed to p99; error metrics include ALL observations.
print('Largest errors for the selected model in 2024:')
display(residuos.sort_values('erro_abs_vencedor_validacao_2023',ascending=False).head(12).round(3))

## 15. Export and Scientific Traceability

We save the complete validation results, the three test comparators, the spatial ablation, the lag-1 temporal analysis, hotel/year-identified predictions, and the executed configuration. **We do not publish hotel names or company-identifying data** in figures or the paper without a confidentiality review.


In [ ]:
ficheiros={
  'resumo_anos':resumo_anos,
  'contagens':contagens,
  'validacao_2023_todas_configuracoes':resultados_validacao,
  'residuos_validacao_2023':diag,
  'teste_2024_comparadores':resultados_teste,
  'ablacao_espacial_teste_2024':resultado_ablacao,
  'experiencia_temporal_lag1':resultados_lag,
  'previsoes_teste_2024':residuos,
  'erros_modelos':pd.DataFrame(falhas,columns=['modelo','politica','tratamento','erro']),
}
for nome,df in ficheiros.items():
    p=PASTA_RESULTADOS/(nome+'.csv')
    df.to_csv(p,index=False)
    print(nome,'→',p)

config={
 'objetivo':'Robustness experiments and the effects of pandemic years; exploratory study',
 'arquivo_entrada':str(FICHEIRO_DADOS.resolve()), 'folha':'New Dataset',
 'anos_treino_validacao':'2018-2022', 'ano_validacao':2023,
 'anos_refit_teste':'2018-2023', 'ano_teste':2024,
 'filtro_independentes':'ht_hotel_chain_d == 0',
 'alvo':COL_ALVO,'features_iniciais':ATRIBUTOS,
 'politicas_examinadas':POLITICAS_2020,'tratamentos_examinados':TRATAMENTOS_ALVO,
 'modelos_executados':modelos_disponiveis,'metrica_selecao':METRICA_SELECAO,
 'configuracao_selecionada_em_2023':{
    'modelo':vencedor['modelo'],'politica_2020':vencedor['politica_2020'],
    'tratamento_alvo':vencedor['tratamento_alvo'],
    'rmse_val':vencedor['val_RMSE'],'mae_val':vencedor['val_MAE']},
 'refit_n':int(len(refit_ate_2023)), 'teste_2024_n':int(len(teste_final)),
 'semente':SEMENTE,
 'nota':'Target outliers capped during training only; validation and test unfiltered; strict t-1 lag available only for eligible hotels.',
 'versoes':{'python':platform.python_version(), 'numpy':np.__version__,
           'pandas':pd.__version__, 'sklearn':__import__('sklearn').__version__},
}
(PASTA_RESULTADOS/'configuracao_execucao.json').write_text(
    json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Export completed. Directory:', PASTA_RESULTADOS.resolve())

## 16. Interpreting the Results Without Overstating Conclusions

**Questions addressed by this notebook**

- Does **excluding 2020** improve, worsen, or preserve 2023 forecasting performance, holding the model and target treatment constant?
- Is **downweighting 2020** more useful than entirely removing observations from that year?
- Do improvements arise mainly from more careful treatment of **training outliers**?
- Do gains observed in 2023 persist in **2024**, after refitting exclusively with earlier data?
- Do spatial features provide predictive information beyond hotel structural characteristics?
- Does productivity at **t−1**, when known, help predict **t**? How well does the persistence baseline perform?

**Limitations and next steps before publication**

1. The models rely mostly on **static variables** and do not incorporate contemporaneous demand/tourism indicators. The pandemic comparison does not replace time-series modelling or establish the causal impact of COVID-19.
2. Model selection uses **a single validation year (2023)** across several combinations. Assess robustness through *rolling-origin backtests* and hotel-level confidence intervals. A one-year horizon may be unstable.
3. The **p99.5** threshold was chosen for convenience; document sensitivity to other percentiles and robust loss functions without selecting on the test set.
4. Productivity may have extreme values due to **small denominators (employee counts)**, accounting differences, and sample-composition changes; these should be audited, not automatically treated as errors.
5. Information on star rating, capacity, location, and restaurants may reflect a survey conducted **after** the forecast year. Verify historical availability to avoid *look-ahead bias*.
6. The **lag-1** experiment applies only to hotels with an observed preceding year; do not generalize its results to unseen hotels.
7. The **2024 test must remain untouched** after its first evaluation. Any variant devised after inspecting the results should be treated as exploratory research and validated again using future data.

**Suggested methodology structure:** Clearly separate (i) panel construction and audit, (ii) the 2020 effect and robust treatment of extreme values, (iii) the temporal protocol and prevention of data leakage, (iv) the added value of spatial features, and (v) limitations of causal interpretation.

**Interpreting contradictory results:** The validation winner does not necessarily win on the test set. If robust training markedly improves performance in 2023 but the advantage disappears in 2024, that is evidence of temporal instability and should be reported rather than concealed.
